In [6]:
!pip install indic-nlp-library

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.1/121.1 kB 2.7 MB/s eta 0:00:00


In [8]:
!python -m indicnlp.download

/usr/bin/python3: No module named indicnlp.download


In [1]:
punctuations = set(['.', ',', '!', '?', '।'])
sentence_enders = set(['.', '?', '!', '।'])

In [5]:
def split_into_words(sentence):
  return sentence.split()

In [12]:
from indicnlp.tokenize import indic_tokenize
from indicnlp.morph import unsupervised_morph

In [26]:
# Morphological analyser for Hindi language, takes input in form of sentence.

def morphological_analysis(hindi_text):
  tokens = list(indic_tokenize.trivial_tokenize(hindi_text, lang='hi'))
  analyzer = unsupervised_morph.UnsupervisedMorphAnalyzer(lang='hi')
  for token in tokens:
        morph_segments = analyzer.morph_analyze(token)

In [34]:
from nltk.tag import tnt
from nltk.corpus import indian
import nltk

In [33]:
nltk.download('indian')

[nltk_data] Downloading package indian to /root/nltk_data...
[nltk_data]   Unzipping corpora/indian.zip.


True

In [37]:
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [38]:
# Hindi POS tagger give input in form of text.

def hindi_model():
    train_data = indian.tagged_sents('/content/hindi.pos')
    tnt_pos_tagger = tnt.TnT()
    tnt_pos_tagger.train(train_data)
    return tnt_pos_tagger

In [148]:
pos_model = hindi_model()
pos_model.tag(nltk.word_tokenize('मैं स्कूल जा रहा हूँ।'))

[('मैं', 'PRP'),
 ('स्कूल', 'NN'),
 ('जा', 'VAUX'),
 ('रहा', 'VAUX'),
 ('हूँ।', 'Unk')]

In [58]:
import pandas as pd
df = pd.read_csv('/content/my_data.csv')
df.head()

,x,y
0,मैं दर्शन करने के लिए भीड़ में से विज्ञान और द...,वे दर्शन करने के लिए भीड़ में से विज्ञान और दर...
1,मन्त्र दीक्षा प्रायः मन्दिर अनुष्ठान जैसे कि प...,मन्त्र दीक्षा प्रायः मन्दिर अनुष्ठान जैसे कि प...
2,मालवा में जीत और हिंदू शासन को बहाल करने के बा...,मालवा में जीत और हिंदू शासन को बहाल करने के बा...
3,"यह लगभग 76 लाख लोग द्वारा बोली जाती है , अर्था...","यह लगभग 76 लाख लोगों द्वारा बोली जाती है , अर्..."
4,इसे धार्मिक संगठनों और राज्य के मध्य व्याप्त प...,यह धार्मिक संगठनों और राज्य के मध्य व्याप्त पा...


In [45]:
df.iloc[0]

'मैं दर्शन करने के लिए भीड़ में से विज्ञान और दर्शन से हमारे पुराने मित्र कवियों के लिए।'

In [55]:
model.tag(nltk.word_tokenize(df.iloc[0]))

[('मैं', 'PRP'),
 ('दर्शन', 'NN'),
 ('करने', 'VNN'),
 ('के', 'PREP'),
 ('लिए', 'PREP'),
 ('भीड़', 'Unk'),
 ('में', 'PREP'),
 ('से', 'PREP'),
 ('विज्ञान', 'Unk'),
 ('और', 'CC'),
 ('दर्शन', 'NN'),
 ('से', 'PREP'),
 ('हमारे', 'PRP'),
 ('पुराने', 'JJ'),
 ('मित्र', 'Unk'),
 ('कवियों', 'Unk'),
 ('के', 'PREP'),
 ('लिए।', 'Unk')]

In [56]:
second_elements = [tag for _, tag in pos_model.tag(nltk.word_tokenize(df.iloc[0]))]

print(second_elements)

['PRP', 'NN', 'VNN', 'PREP', 'PREP', 'Unk', 'PREP', 'PREP', 'Unk', 'CC', 'NN', 'PREP', 'PRP', 'JJ', 'Unk', 'Unk', 'PREP', 'Unk']


In [63]:
correct = df['x']
incorrect = df['y']

In [83]:
df = df.rename(columns={'x': 'sentence'})
correct = df['sentence']

In [84]:
import numpy as np
correct = pd.DataFrame(correct)
correct['output'] = np.ones(5696)

In [87]:
incorrect = pd.DataFrame(incorrect)
incorrect = incorrect.rename(columns={'y': 'sentence'})
incorrect['output'] = np.zeros(5696)

In [90]:
hf = pd.concat([correct, incorrect])

In [92]:
hf = hf.sample(frac=1).reset_index(drop=True)

In [95]:
hf['POS_tags'] = hf['sentence'].apply(lambda x: [tag for _, tag in model.tag(nltk.word_tokenize(x))])

In [98]:
hf = hf.drop('sentence', axis=1)

In [102]:
import ast
hf['POS_tags'] = hf['POS_tags'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

In [147]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from torch.nn.utils.rnn import pad_sequence

In [135]:
all_pos_tags = set(pos for tags in hf['POS_tags'] for pos in tags)
pos_encoder = LabelEncoder()
pos_encoder.fit(list(all_pos_tags))

LabelEncoder()

In [136]:
def encode_pos_tags(pos_tags):
    return torch.tensor(pos_encoder.transform(pos_tags), dtype=torch.long)

hf['encoded_tags'] = hf['POS_tags'].apply(encode_pos_tags)

In [137]:
train_hf, test_hf = train_test_split(hf, test_size=0.2, random_state=42)

In [138]:
train_hf = train_hf.reset_index(drop=True)
test_hf = test_hf.reset_index(drop=True)

In [139]:
class POSDataset(Dataset):
    def __init__(self, encoded_tags, labels):
        self.encoded_tags = encoded_tags
        self.labels = labels.values.astype('float32')

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.encoded_tags[idx], torch.tensor(self.labels[idx])

In [140]:
def collate_fn(batch):
    sequences, labels = zip(*batch)
    sequences_padded = pad_sequence(sequences, batch_first=True)
    return sequences_padded, torch.tensor(labels)

In [141]:
train_dataset = POSDataset(train_hf['encoded_tags'], train_hf['output'])
test_dataset = POSDataset(test_hf['encoded_tags'], test_hf['output'])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=32, collate_fn=collate_fn)

In [153]:
class LSTMModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super(LSTMModel, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.embedding(x)
        _, (h_n, _) = self.lstm(x)
        out = self.fc(h_n[-1])
        return self.sigmoid(out).squeeze()

    def predict(self, pos_tag_seq, threshold=0.5):
        self.eval()
        if isinstance(pos_tag_seq, list):
            pos_tag_seq = torch.tensor(pos_tag_seq, dtype=torch.long).unsqueeze(0)
        elif isinstance(pos_tag_seq, torch.Tensor) and len(pos_tag_seq.shape) == 1:
            pos_tag_seq = pos_tag_seq.unsqueeze(0)

        with torch.no_grad():
            output = self.forward(pos_tag_seq)
            return float(output.item() > threshold)

In [155]:
vocab_size = len(pos_encoder.classes_)
embedding_dim = 64
hidden_dim = 128

model = LSTMModel(vocab_size, embedding_dim, hidden_dim)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.0005)

In [156]:
for epoch in range(10):
    model.train()
    total_loss = 0
    for inputs, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

In [158]:
text = 'मुझे किताबें पढ़ना बहुत पसंद है।'
second_elements = [tag for _, tag in pos_model.tag(nltk.word_tokenize(text))]
encoded = torch.tensor(pos_encoder.transform(second_elements), dtype=torch.long)
model.predict(encoded)

1.0

In [159]:
import pickle

filename = 'grammer_model.pkl'
pickle.dump(model, open(filename, 'wb'))